In [2]:
! pip install peft

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   ---------------------------------------- 557.0/557.0 kB 2.4 MB/s  0:00:00


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Define paths
BASE_MODEL = "Qwen/Qwen2.5-3B"
ADAPTER_PATH = "checkpoint-250"

In [4]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Ensure the ChatML template matches your training setup
tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

Loading tokenizer...


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
print("Loading base model in 4-bit...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

Loading base model in 4-bit...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [6]:
print("Applying LoRA adapter...")
# This layers your fine-tuned weights on top of the base model
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

Applying LoRA adapter...


In [7]:
# Setup a conversation
messages = [
    {"role": "system", "content": "You are a helpful and precise assistant."},
    {"role": "user", "content": "Explain how multi-head attention works in a Transformer."}
]

# Format the prompt using the ChatML template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Generating response...\n")
inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

Generating response...



In [8]:
# Generate parameters (tweak top_p, top_k, and temperature as needed)
outputs = model.generate(
    **inputs, 
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

# Decode and slice out only the new generated text
input_length = inputs.input_ids.shape[-1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [9]:
print("--- Output ---")
print(response)

--- Output ---
Multi-head attention is a key component of the Transformer model, which was introduced by Vaswani et al. in their 2017 paper. It allows the model to attend to multiple different representations of the same input, rather than just the single representation used in traditional attention mechanisms.

In a multi-head attention mechanism, the input is divided into multiple "heads," each of which computes a separate attention score matrix. These heads are then concatenated together and passed through a linear layer to produce the final output. The attention score matrices are typically normalized using the softmax function to ensure that they sum to one.

The main advantage of using multiple heads is that it allows the model to attend to different parts of the input sequence, rather than just the most important parts. This can be particularly useful for tasks such as language translation, where the input sequence may contain important information that is not immediately obviou